In [1]:
#import libraries
import pyodbc
import pandas as pd 
import numpy as np 
import warnings
from scipy import stats
from scipy.stats import chi2_contingency
warnings.filterwarnings("ignore") 


In [2]:
# set connection

conn = pyodbc.connect(
        "Driver={SQL Server};"
        "Server=LocalHost\\SQLEXPRESS;"
        "Database=BankChurn;"
        "Trusted_Connection=yes;"
    )

cursor = conn.cursor()
print("Connection Successfull")

Connection Successfull


# Test 1 : Independent Sample t-test

## Business Question 
In the banking industry, account balance is one of the key indicators of customer value. The business wants to understand:


 * Do customers who leave the bank (churn) differ in accounts balance compared to loyal customers?

The answer to this question directly influences several startegic descisions :
  * Retention Campaign Targeting
  * VIP Protection Programs
  * Risk Segmentation
  * Revenue Forecasting

So, we can say:

- **If high-balance customers are more likely to churn:** There is a significant risk of losing high-value clients, requiring immediate retention and protection strategies.
- **If low-balance customers churn more frequently:** This may indicate issues with customer engagement, product fit, or value proposition.

# Statistical Hypotheses

We aim to compare the **average account balance** between two customer groups:

- Customers who **churned**
- Customers who have been **retained (loyal)**

### Null Hypothesis (H₀) :
The **mean account balance is equal** between the two groups.

### Alternative Hypothesis (H₁) :
The **mean account balance differs** between the two groups.

# Test Assumptions

The Independent Samples t-test relies on several important assumptions:

- **Independence:** Observations must be independent of one another. In banking datasets, each customer represents a unique record, so this assumption is satisfied.

- **Normality:** The distribution of **Balance** within each group should be approximately normal. If the sample size is large (**n > 30 per group**), the t-test is considered relatively robust to normality violations due to the Central Limit Theorem.

- **Homogeneity of Variance:** The variance of **Balance** should be equal across the two groups. This assumption can be tested using **Levene’s Test**. If the variances are unequal, the appropriate alternative is **Welch’s t-test**.

# Why this test?

The choice of statistical test is based on the structure of the variables and the analytical objective:

- **Dependent variable is Numerical** (`Balance`)
- **Independent variable is Binary categorical** (`Churned`)
- **Objective is comparing group means**

## Test Implementation

 * Step 1: Data Preperation 

In [3]:
query = """ 
select 
a.Balance, 
d.Churned
from Demographic  as d 
join Account as a 
on d.CustomerId = a.CustomerId

"""

df_t_test = pd.read_sql(query, conn)
df_t_test.head()

,Balance,Churned
0,119827.49,True
1,83807.86,False
2,159660.80,True
3,119827.49,False
4,125510.82,False


 * Step 2: Split Groups  

In [4]:
churned_filt = df_t_test["Churned"] == 1
retained_filt = df_t_test["Churned"] == 0

churned_t_test = df_t_test.loc[churned_filt, "Balance"]
retained_t_test = df_t_test.loc[retained_filt , "Balance"]

In [5]:
churned_t_test

0       119827.49
2       159660.80
5       113755.78
7       115046.74
16      132602.88
          ...    
9981    152039.70
9982    137145.12
9991     88381.21
9997    119827.49
9998     75075.31
Name: Balance, Length: 2037, dtype: float64

 *  Step 3: Descriptive Stats

In [6]:
print("Churned Mean:" , churned_t_test.mean())
print("Retained Mean:" ,  retained_t_test.mean())








Churned Mean: 120521.27620520373
Retained Mean: 119650.01690066556


 * Step 4: Variance Check

In [7]:
# if p > 0.05 , equal variance assumed
levene_test  = stats.levene(churned_t_test , retained_t_test)
print(f"P-value: {levene_test.pvalue:.11f}")

P-value: 0.00000000143


### Since p < 0.05, Null Hypothesis is rejected,
#### there is a difference between balance means and it is not by pure chance, the difference is stastically significant 

#### since the variances are not equal we use the Welch's t-test

In [8]:


t_stat , p_value = stats.ttest_ind(
    churned_t_test,
    retained_t_test,
    equal_var=False
)

print("t_stat:", t_stat)
print("p_value:", p_value)


t_stat: 1.3534724523393942
p_value: 0.17601045694361078


## Output Interpretation
**p-value is greater than 0.05 , we fail to reject the null hypothesis, there is no statistical evidence that the accounts balance affects churn activity**

# Test 2: Chi-Square Test of Independence

## Business Question 
In the banking industry, one of the strongest signals of churn risk is account activity. The business wants to understand:


 * Are inactive users more likely to churn?

if there is a realtionship between inactivity and churning, then it can directly support starategic intiatives like :
  * Early Warning System Developement
  * Reactivation Campaign Design 
  * Customer Health Score Modeling


if customer acitivity is associated with churn:

- **Enagagement becomes a key driver of churn** .
- **Behaviour based retention strategies will be required to reduce it.** 

# Statistical Hypotheses

We are examining the relationship between two categorical variables:

- `Churned` → Yes / No
- `IsActive` → Active / Inactive

- **Null Hypothesis:** Churn and customer activity are independent.
- **Alternative Hypothesis:** Churn and customer activity are associated (dependent).

# Column Selection

To perform this hypothesis test, only two columns are required:

- `IsActive` represents whether the account is still active or not.
- `Churned` defines the two customer groups (churned vs. retained) used for the statistical comparison.

# Contingency Table Concept

The Chi-Square test operates on a frequency (contingency) table.

It evaluates whether the observed frequencies across categorical groups differ significantly from the frequencies we would expect if the variables were independent.

# Test Assumptions

To ensure the validity of the Chi-Square test, the following assumptions must be satisfied:

- **Variables Are Categorical:** Both variables must be categorical.
- **Independence of Observations:** Each observation must be independent. In the banking dataset, each customer represents a unique record, so this assumption holds.
- **Expected Frequency > 5:** Each cell in the contingency table should have an expected frequency of at least 5. If this condition is not met, we use an **Exact Test**.

# Why this test?

The choice of statistical test is based on the structure of the variables and the analytical objective:

- **Dependent variable is Categorical** (`IsActive`)
- **Independent variable is Binary categorical** (`Churned`)
- **Objective is association**

## Test Implementation

 * Step 1: Data Preperation 

In [9]:
query = """ 
select 
a.isActive, 
d.Churned
from Demographic  as d 
join Account as a 
on d.CustomerId = a.CustomerId

"""
df_chi = pd.read_sql(query,conn)
df_chi.head()

,isActive,Churned
0,True,True
1,True,False
2,False,True
3,False,False
4,True,False


* Step 2: Build Contingency Table

In [10]:
ct = pd.crosstab(df_chi["Churned"] , df_chi["isActive"])
ct

isActive,False,True
Churned,,
False,3547,4416
True,1302,735


In [11]:
# there are about 3500 customers who are not active and they did not churn 
# there are about 4416 customers who are not active and they churned 
# there are about 1302 customers who are active and they did not churn  
# there are about 735 customers who are active and they churned  



 * Step 3: Run the chi-Square test

In [12]:
chi2 , p_value , dof , expected = chi2_contingency(ct)

print("Chi2:    " , chi2)
print("p_value: " ,  p_value )
print("dof:     " , dof)
print("expected:" , expected)


Chi2:     242.98534164287963
p_value:  8.785858269303703e-55
dof:      1
expected: [[3861.2587 4101.7413]
 [ 987.7413 1049.2587]]


# Output Interpretation

**The p-value is less than 0.05. It means we reject the null hypothesis of the test. So, the final interpretation is that churn and customer activity are associated (dependent)**.

# Test 3: ANOVA 

## Business Question 
The Bank Operates across multiple countries/regions. The business wants to understand:


 * Does Customer tenure differ significanlty across countries?

if the tenure is lower in certain countries, it may indicate that :
  * Customers in those markets tend to leave earlier.
  * Service Quality,Product Offerings or digital experience may be weaker in those regions.
  * that specific market is highly competitive.
  * Customer aquisition and attrition dynamics differ by market.

The result of this test can help drive strategic decisions such as,


- Designing country-specific retention strategies
- Increasing investment in products or channels in markets with lower tenure
- Benchmarking regional and branch performance across countries

# Statistical Hypotheses

- **Null Hypothesis:** The average tenure is the same across all countries.
- **Alternative Hypothesis:** The average tenure is not the same across all countries.

# Column Selection

To perform this hypothesis test, only two columns are required:

- **Tenure** as the numerical column, which shows how long the client has been using the services.
- **Geography** as the multiple categorical column, which shows different locations.

# Test Assumptions

To ensure the validity of the ANOVA test, the following assumptions should be evaluated:

- **Independence:** Each customer must represent an independent observation. In customer datasets, this assumption is typically satisfied, as each customer is recorded once.
- **Normality Within Each Group:** The distribution of Tenure within each country should be approximately normal (or sample sizes should be sufficiently large). In practice, ANOVA is relatively robust to normality violations, particularly when each group has a large number of observations.
- **Homogeneity of Variances:** The variance of Tenure should be reasonably similar across countries. This assumption can be assessed using **Levene's Test**.
- **Sample Size per Group:** Each country should have an adequate number of observations.

# Why this test?

The choice of statistical test is based on the structure of the variables and the analytical objective:

- **Dependent variable is Numerical** (`Tenure`)
- **Independent variable is Multi-categorical** (`Location`)
- **Objective is comparing average means**

## Test Implementation

 * Step 1: Data Preperation 

In [15]:
query = """ 
select 
     a.Tenure ,
     l.Geography as Country
from account as a 
join demographic as d
     on d.CustomerId = a.CustomerId
join location as l 
     on d.LocationId = l.LocationId
where a.Tenure is not null
and l.Geography is not null
"""
df_anova = pd.read_sql(query,conn)
df_anova.head()

,Tenure,Country
0,9,Spain
1,1,Germany
2,1,USA
3,6,Spain
4,10,Spain


 * Step 2: Descriptve stats

In [20]:
summary  = df_anova.groupby("Country")["Tenure"].agg(["count" , "mean" , "std"])
print(summary.sort_values("mean" , ascending= False))

         count      mean       std
Country                           
UK        1703  5.127422  2.939195
France    1714  5.042590  2.904582
USA       1669  5.002397  2.890694
Germany   1626  4.987700  2.870085
Canada    1679  4.982132  2.888003
Spain     1609  4.927906  2.857046


 * Step 3:Check Variance Quality

In [49]:
# if p-vlaue > 0.05, equal variance is assumed
groups = []

for country, g in df_anova.groupby("Country"):
    groups.append(g["Tenure"].values)
levene_test = stats.levene(*groups)
levene_test

### since p-value is greater than 0.05, null hypothesis is rejected, so vairances are not equal,we use the welch's anova test instead.

* Step 3: Run Welch's Anova Test

In [52]:
f_stat, p_value_anova = stats.f_oneway(*groups)
print("F:" , f_stat)
print("p-value:" , p_value_anova)

F: 0.9151410065439716
p-value: 0.46986633791017374


# Output Interpretation

**The p-value is greater than 0.05, so we fail to reject the null hypothesis. We do not have sufficient statistical evidence to conclude that the average tenure differs across countries.**

**Meaning we can't really say that customer tenure differs significantly across the countries**

# Test 3: EFFECT SIZE

so for the results from our tests are,
 * `welch's t-test`: There is no significant balace mean difference between churned and retained customers.
 * `chi-square`: There is a significant association between account acitivity and churning
 * `Anova`: No significant tenure mean difference between different countries

we also need find hwo big is the impact of each test


### Effect size of t-test - cohen's d

### Cohen's d — Effect Size

- **0.2** --> Small effect
- **0.5** --> Medium effect
- **0.8** --> Large effect
- **1.2+** --> Very large effect

In [56]:
# since the sizes of the both groups are uequal , chonen's d formula is used

# 1. Calculate the means
churned_t_test_mean = churned_t_test.mean()
retained_t_test_mean = retained_t_test.mean()

# 2. Calculate the standard deviations
churned_t_test_std = churned_t_test.std()
retained_t_test_std = retained_t_test.std()

# 3. Calculate sample sizes
churned_t_test_n = len(churned_t_test)
retained_t_test_n = len(retained_t_test)

# 4. Calculate pooled standard deviation
pooled_std = np.sqrt(
    (
        (churned_t_test_n - 1) * churned_t_test_std**2
        + (retained_t_test_n - 1) * retained_t_test_std**2
    )
    / (churned_t_test_n + retained_t_test_n - 2)
)

# 5. Calculate Cohen's d
cohen_d = (churned_t_test_mean - retained_t_test_mean) / pooled_std

print("Cohen's d:", cohen_d)

Cohen's d: 0.03623904081079434


# Output Interpretation

**The Independent Samples t-test indicated no statistically significant difference in average account balance between churned and retained customers** (high p-value).

**Additionally, the effect size (Cohen's d = 0.03) is negligible, suggesting that account balance has minimal practical impact on churn behavior**.

### Key Findings

- **High p-value in t-test:** The difference in average balance between churned and retained customers is not statistically significant.
- **Small Cohen's d:** The difference is negligible from a practical/business perspective.

# Cramér's V : Chi-Square



### Cramér's V — Effect Size

- **0.1** --> Weak
- **0.3** --> Moderate
- **0.5** --> Strong

In [57]:
n_chi = ct.sum().sum()
k_chi = min(ct.shape)

cramers_v = np.sqrt(chi2 / (n_chi * (k_chi - 1)))

print("Cramér's V:", cramers_v)

Cramér's V: 0.15587987094005423


# Output Interpretation

The Chi-Square test shows a statistically significant relationship between customer activity and churn. However, **Cramér's V = 0.15** indicates a weak association, suggesting that while engagement is related to churn, it is not the sole driving factor.

### Key Findings

- **Small p-value in Chi-Square test:** There is a statistically significant relationship between active account status and churn.
- **Cramér's V = 0.15:** The association is weak, suggesting that customer activity is related to churn but is not the only factor influencing churn.

# Eta/Omega : ANOVA

The below table shows the result for interpretation:

### Eta/Omega — Effect Size

- **0.01** --> Small
- **0.06** --> Medium
- **0.14** --> Large

In [59]:
# Calculate SS
all_values = df_anova["Tenure"].values
grand_mean = np.mean(all_values)

ss_between = sum(
    len(group) * (np.mean(group) - grand_mean)**2
    for group in groups
)

ss_within = sum(
    sum((group - np.mean(group))**2)
    for group in groups
)

ss_total = ss_between + ss_within
print("ss_total:" ,ss_total)

ss_total: 83638.36159999996


In [60]:
# Calculate Eta score
k = len(groups)
n = len(all_values)

df_between = k - 1
df_within = n - k

ms_within = ss_within / df_within

eta_squared = ss_between / ss_total
print("Eta Squared (η²):", eta_squared)

Eta Squared (η²): 0.00045763568409215794


In [61]:
# Calculate Omega score
omega_squared = (
    ss_between - (df_between * ms_within)
) / (ss_total + ms_within)

print("Omega Squared (ω²):", omega_squared)


Omega Squared (ω²): -4.2431297066592226e-05


# Output Interpretation

The one-way ANOVA test indicates no statistically significant difference in average tenure across countries (p-value > 0.05). Effect size measures further confirm this finding, with Eta Squared (0.00046) and Omega Squared (0) indicating a negligible practical impact. This suggests that geographic region does not meaningfully influence customer tenure in this dataset.

### Key Findings

- **High p-value in ANOVA test:** No statistically significant difference in average tenure across countries.
- **Small Omega/Eta:** Even if there were a difference, that difference would be negligible from a practical perspective.